# MetalDAM → PSCL Finetuning — Kaggle

| Stage | What it does | Output |
|-------|-------------|--------|
| 1 — Self-supervised pretraining | Contrastive learning on 788 unlabelled patches | `moco200.pt` |
| 2 — Supervised finetuning | 4-class segmentation on labelled patches | `fine.pt` |

**Before running:** Add the `pscl-files` dataset via the Kaggle **Add Data** panel.

**Resume support:** set `RESUME_DATASET` in Cell 0 to restore checkpoints from a previous session.

---
## Step 0 — Add your dataset on Kaggle

In the Kaggle notebook editor click **Add Data** and attach `alirezahorri/pscl-files`.
It will be mounted read-only at:
```
/kaggle/input/datasets/alirezahorri/pscl-files/
├── pscl_fintune_code/PSCL/   ← PSCL source
└── metaldam_patches/          ← MetalDam/data/patches/
```

In [ ]:
# ── USER CONFIGURATION ───────────────────────────────────────────────────────

BASE = '/kaggle/input/datasets/alirezahorri/pscl-files'

PRETRAIN_EPOCHS = 200
GPU = '0'

# Path to a Kaggle dataset with checkpoints from a previous session.
# Leave '' on first run. After a session: download moco*.pt from the Output tab,
# upload as a Kaggle dataset, add it here (e.g. '/kaggle/input/pscl-checkpoints').
RESUME_DATASET = ''

---
## Cell 1 — Clone code from GitHub and copy PSCL source

Run once per session. Already-present files are skipped.

In [ ]:
import os, subprocess, sys, shutil

WORK = '/kaggle/working'

# ── 1. Clone Pscl_fintune from GitHub ────────────────────────────────────────
if not os.path.exists(f'{WORK}/repo'):
    subprocess.run([
        'git', 'clone',
        '-b', 'kaggle',
        'https://github.com/arhorri/PSCL_fintune.git',
        f'{WORK}/repo'
    ], check=True)
    print('Cloned.')
else:
    subprocess.run(['git', '-C', f'{WORK}/repo', 'pull'], check=True)
    print('Pulled latest.')

if not os.path.exists(f'{WORK}/Pscl_fintune'):
    shutil.copytree(f'{WORK}/repo/PSCL_fintune', f'{WORK}/Pscl_fintune')
    print('Pscl_fintune ready.')

# ── 2. Copy PSCL source from dataset (read-only → working) ───────────────────
if not os.path.exists(f'{WORK}/PSCL'):
    shutil.copytree(f'{BASE}/pscl_fintune_code/PSCL', f'{WORK}/PSCL')
    print('PSCL source ready.')
else:
    print('PSCL already present.')

# ── 3. Restore checkpoints from previous session (if RESUME_DATASET set) ─────
if RESUME_DATASET:
    ckpt_src = os.path.join(RESUME_DATASET, 'self_UNet_metaldam', '_Numf', 'f')
    ckpt_dst = os.path.join(WORK, 'Pscl_fintune', 'self_UNet_metaldam', '_Numf', 'f')
    if os.path.exists(ckpt_dst):
        print('Checkpoints already present.')
    elif os.path.exists(ckpt_src):
        shutil.copytree(ckpt_src, ckpt_dst)
        ckpts = sorted(f for f in os.listdir(ckpt_dst) if f.endswith('.pt'))
        print(f'Restored {len(ckpts)} checkpoint(s): {ckpts}')
    else:
        print(f'WARNING: no checkpoints found at {ckpt_src}')

print('Setup complete.')

---
## Cell 2 — Set paths and verify environment

In [ ]:
import sys, os
import torch

CODE_DIR = '/kaggle/working/Pscl_fintune'
PSCL_SRC = '/kaggle/working/PSCL/PSCL'
DATA_DIR = f'{BASE}/metaldam_patches/MetalDam/data/patches'

for p in [PSCL_SRC, CODE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

os.chdir(CODE_DIR)

print(f'Working dir : {os.getcwd()}')
print(f'PSCL source : {PSCL_SRC}')
print(f'Data dir    : {DATA_DIR}')
print(f'CUDA        : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {props.total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU. Go to Settings > Accelerator > GPU T4.')

---
## Cell 3 — Verify MetalDAM data

In [ ]:
import os

splits = {'train': 788, 'val': 168, 'test': 192}
all_ok = True

print('Verifying MetalDAM patches...')
for split, expected in splits.items():
    img_dir  = os.path.join(DATA_DIR, split, 'images_norm')
    mask_dir = os.path.join(DATA_DIR, split, 'masks')
    n_img  = len([f for f in os.listdir(img_dir)  if f.endswith('.png')]) if os.path.exists(img_dir)  else 0
    n_mask = len([f for f in os.listdir(mask_dir) if f.endswith('.png')]) if os.path.exists(mask_dir) else 0
    ok = (n_img == expected and n_mask == expected)
    mark = 'OK  ' if ok else 'FAIL'
    print(f'  [{mark}] {split:5s}: {n_img} images, {n_mask} masks  (expected {expected})')
    if not ok:
        all_ok = False

if all_ok:
    print('\nAll splits verified. Ready to train.')
else:
    print('\nData check FAILED — re-check your zip file and re-upload.')

---
## Cell 4 — Detect checkpoints

Scans `/kaggle/working/` for existing Stage 1 checkpoints so training can resume.

In [ ]:
import glob, re, os

SELF_CKPT_DIR = os.path.join(CODE_DIR, 'self_UNet_metaldam', '_Numf', 'f')
FINE_CKPT_DIR = os.path.join(CODE_DIR, 'fine_UNet_metaldam', '_Numf', 'f')

for label, d in [('self_UNet_metaldam/_Numf/f', SELF_CKPT_DIR),
                  ('fine_UNet_metaldam/_Numf/f', FINE_CKPT_DIR)]:
    print(f'{"Present" if os.path.exists(d) else "Not found"}: {label}')

ckpts = sorted(glob.glob(os.path.join(SELF_CKPT_DIR, 'moco*.pt')))
if ckpts:
    latest = ckpts[-1]
    START_EPOCH = int(re.search(r'moco(\d+)\.pt', os.path.basename(latest)).group(1))
    RESUME_CKPT = latest
    if START_EPOCH >= PRETRAIN_EPOCHS:
        print(f'\nStage 1 already complete (epoch {START_EPOCH}).')
    else:
        print(f'\nLatest checkpoint: epoch {START_EPOCH} — will resume from here.')
else:
    START_EPOCH = 0
    RESUME_CKPT = None
    print('\nNo Stage 1 checkpoint found — will start from scratch.')

print(f'START_EPOCH = {START_EPOCH}')
print(f'RESUME_CKPT = {RESUME_CKPT}')

---
## Cell 5 — Stage 1: Self-supervised pretraining

PSCL trains a UNet encoder using contrastive learning: the 788 unlabelled training
patches form the pool; the 168 labelled val patches guide semantically-aware patch
sampling.

**Expected duration on Colab T4:**
- First batch: ~3–5 min (CUDA kernel compilation, once per session)
- Per batch after warmup: ~5–15 s
- Per epoch (~98 batches): ~10–25 min
- 200 epochs: ~1–3 days total

**Checkpoints** are saved every 10 epochs to `self_UNet_metaldam_Numf/f/moco{N}.pt`.
Back them up with Cell 6 after each session.

**Progress** is printed every 10 batches:
```
Training: epoch 0 → 200, 98 batches/epoch
Epoch [0/200] [1/98] Loss 4.23  C 1.12  Dense 2.11  lr 0.0001  batch 8.4s
```

In [ ]:
import config_metaldam

if START_EPOCH >= PRETRAIN_EPOCHS:
    print(f'Stage 1 already complete at epoch {START_EPOCH}. Skipping.')
    print('To re-run, delete self_UNet_metaldam_Numf/ and reset START_EPOCH = 0.')
else:
    remaining = PRETRAIN_EPOCHS - START_EPOCH
    print(f'Running Stage 1: epochs {START_EPOCH} to {PRETRAIN_EPOCHS} ({remaining} remaining)')
    print('-' * 60)
    config_metaldam.run(
        method='self',
        tt='metaldam',
        data_dir=DATA_DIR,
        self_max_epoch=PRETRAIN_EPOCHS,
        start_epoch=START_EPOCH,
        resume_ckpt=RESUME_CKPT,
        env=GPU,
    )
    print('\nStage 1 complete.')

---
## Cell 6 — List Stage 1 checkpoints

Download from the **Output** tab to keep across sessions.

In [ ]:
import os

src = os.path.join(CODE_DIR, 'self_UNet_metaldam', '_Numf', 'f')
if not os.path.exists(src):
    print('Nothing saved yet — Stage 1 has not run.')
else:
    ckpts = sorted(f for f in os.listdir(src) if f.endswith('.pt'))
    print(f'{len(ckpts)} checkpoint(s) in: {src}')
    for name in ckpts:
        print(f'  {name:<20s}  {os.path.getsize(os.path.join(src, name))/1e6:.0f} MB')
    print('\nDownload from the Kaggle Output tab to keep across sessions.')

---
## Cell 7 — Stage 2: Supervised finetuning

Loads the encoder from `moco200.pt`, attaches a fresh decoder, and finetunes for
50 epochs on the labelled training patches.

- Encoder LR: `1e-4` (small — preserves pretrained features)
- Decoder LR: `1e-3` (large — trains from scratch)
- Loss: `0.5 × BCE (weighted [1,1,5,5]) + 0.5 × Dice`
- Val accuracy checked every 10 epochs; best model saved as `fine.pt`

**Expected duration on Colab T4:** ~1–3 hours.

In [ ]:
import config_metaldam, os

fine_ckpt = os.path.join(CODE_DIR, 'fine_UNet_metaldam', '_Numf', 'f', 'fine.pt')
moco_ckpt = os.path.join(CODE_DIR, 'self_UNet_metaldam', '_Numf', 'f', f'moco{PRETRAIN_EPOCHS}.pt')

if os.path.exists(fine_ckpt):
    print('fine.pt already exists — skipping Stage 2.')
elif not os.path.exists(moco_ckpt):
    print(f'ERROR: Stage 1 checkpoint not found: {moco_ckpt}')
    print('Complete Stage 1 first (Cell 5).')
else:
    print(f'Running Stage 2 from: {moco_ckpt}')
    print('-' * 60)
    config_metaldam.run(
        method='fine',
        tt='metaldam',
        data_dir=DATA_DIR,
        load_moco_ep=str(PRETRAIN_EPOCHS),
        env=GPU,
    )
    print('\nStage 2 complete.')

---
## Cell 8 — List all output files

Download from the **Output** tab.

In [ ]:
import os

for label, path in [
    ('self_UNet_metaldam/_Numf', os.path.join(CODE_DIR, 'self_UNet_metaldam', '_Numf')),
    ('fine_UNet_metaldam/_Numf', os.path.join(CODE_DIR, 'fine_UNet_metaldam', '_Numf')),
]:
    print(f'{"Present" if os.path.exists(path) else "Not found"}: {path}')

print('\nAll outputs are in /kaggle/working/ — download from the Output tab.')

---
## Cell 9 — Training curves

Parses the log files and plots pretraining loss + finetuning loss + validation metrics.

In [ ]:
import re, os
import matplotlib.pyplot as plt

def _parse_self(path):
    eps, loss, c_loss, d_loss = [], [], [], []
    if not os.path.exists(path):
        return eps, loss, c_loss, d_loss
    seen = set()
    with open(path) as f:
        for line in f:
            m = re.search(r'Epoch \[(\d+)/\d+\].*Loss ([\d.]+).*C ([\d.]+).*Dense ([\d.]+)', line)
            if m:
                ep = int(m.group(1))
                seen.add(ep)
                if eps and eps[-1] == ep:
                    loss[-1], c_loss[-1], d_loss[-1] = float(m.group(2)), float(m.group(3)), float(m.group(4))
                else:
                    eps.append(ep); loss.append(float(m.group(2)))
                    c_loss.append(float(m.group(3))); d_loss.append(float(m.group(4)))
    return eps, loss, c_loss, d_loss

def _parse_fine(path):
    eps, loss, v_acc, v_miou = [], [], [], []
    if not os.path.exists(path):
        return eps, loss, v_acc, v_miou
    with open(path) as f:
        for line in f:
            m = re.search(r'Epoch \[(\d+)/\d+\] Loss ([\d.]+)', line)
            if m:
                eps.append(int(m.group(1))); loss.append(float(m.group(2)))
            m2 = re.search(r'\[val\] ACC ([\d.]+).*mIoU ([\d.]+)', line)
            if m2:
                v_acc.append(float(m2.group(1))); v_miou.append(float(m2.group(2)))
    return eps, loss, v_acc, v_miou

self_log = os.path.join(CODE_DIR, 'self_UNet_metaldam_Numf', 'f', 'log_self.txt')
fine_log = os.path.join(CODE_DIR, 'fine_UNet_metaldam_Numf', 'f', 'log_fine.txt')

eps_s, loss_s, c_s, d_s   = _parse_self(self_log)
eps_f, loss_f, v_acc, v_mi = _parse_fine(fine_log)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Training curves', fontsize=13)

if eps_s:
    axes[0].plot(eps_s, loss_s, 'b-',  lw=1.5, label='Total')
    axes[0].plot(eps_s, c_s,    'g--', lw=1,   label='Global InfoNCE')
    axes[0].plot(eps_s, d_s,    'r--', lw=1,   label='Dense patch')
    axes[0].set_title('Stage 1 — Pretraining loss')
    axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No Stage 1 log yet', ha='center', va='center', transform=axes[0].transAxes)

if eps_f:
    axes[1].plot(eps_f, loss_f, 'b-', lw=1.5)
    axes[1].set_title('Stage 2 — Finetuning loss')
    axes[1].set_xlabel('Epoch'); axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No Stage 2 log yet', ha='center', va='center', transform=axes[1].transAxes)

if v_acc:
    x = list(range(len(v_acc)))
    axes[2].plot(x, v_acc,  'g-o', ms=5, label='Val Accuracy')
    axes[2].plot(x, v_mi,   'r-o', ms=5, label='Val mIoU')
    axes[2].set_title('Stage 2 — Validation metrics')
    axes[2].set_xlabel('Eval step'); axes[2].legend(); axes[2].grid(alpha=0.3)
else:
    axes[2].text(0.5, 0.5, 'No val metrics yet', ha='center', va='center', transform=axes[2].transAxes)

plt.tight_layout()
out_path = os.path.join('/kaggle/working', 'training_curves.png')
plt.savefig(out_path, dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved to {out_path}')

---
## Cell 10 — Visualise test-split predictions

Loads the best finetuned model (`fine.pt`) and shows 4 random test patches
with ground-truth and predicted segmentation masks side by side.

In [ ]:
import sys, os, random
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch

for p in [PSCL_SRC, CODE_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from model import UNet
from data_metaldam import MetalDAMDataset

PSCL_COLORS = {
    0:   ( 43, 255,   0),
    1:   (128,   0, 255),
    2:   (255, 255,   0),
    3:   (255,   0,   0),
    255: ( 30,  30,  30),
}
PSCL_NAMES = ['Austenite (0)', 'Matrix (1)', 'MA (2)', 'Precipitate (3)', 'Ignore (255)']
MEAN = np.array([0.49139968, 0.48215841, 0.44653091])
STD  = np.array([0.24703223, 0.24348513, 0.26158784])

fine_ckpt = os.path.join(CODE_DIR, 'fine_UNet_metaldam_Numf', 'f', 'fine.pt')

if not os.path.exists(fine_ckpt):
    print('fine.pt not found — run Stage 2 (Cell 7) first.')
else:
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = UNet(drop=0, channel=[32, 64, 128, 256],
                 IncNorm=['BN', 'BN'], DownNorm=['BN', 'BN'], UpNorm=['BN', 'LN'])
    ckpt_data = torch.load(fine_ckpt, map_location='cpu')
    model.load_state_dict(ckpt_data['finetune'])
    model = model.to(device).eval()
    print(f'Loaded fine.pt  epoch={ckpt_data["epoch"]}  val_acc={ckpt_data["acc"]:.4f}')

    test_ds = MetalDAMDataset(DATA_DIR, split='test')
    n_show  = min(4, len(test_ds))
    indices = random.sample(range(len(test_ds)), n_show)

    def colorise(mask):
        rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
        for idx, color in PSCL_COLORS.items():
            rgb[mask == idx] = color
        return rgb

    def unnorm(t):
        img = t.numpy().transpose(1, 2, 0) * STD + MEAN
        return np.clip(img, 0, 1)

    fig, axes = plt.subplots(3, n_show, figsize=(4 * n_show, 9))
    fig.suptitle('Test-split predictions — finetuned PSCL', fontsize=12)
    for row, label in enumerate(['Image', 'Ground truth', 'Prediction']):
        axes[row, 0].set_ylabel(label, fontsize=11)

    with torch.no_grad():
        for col, idx in enumerate(indices):
            img_t, gt_t, fname = test_ds[idx]
            pred = model(img_t.unsqueeze(0).to(device))
            pred_cls = torch.softmax(pred, dim=1).cpu().numpy()[0]
            pred_cls = np.argmax(pred_cls, axis=0).astype(np.uint8)

            gt_np  = gt_t.numpy()
            gt_cls = np.argmax(gt_np, axis=0).astype(np.uint8)
            gt_cls[gt_np.sum(axis=0) == 0] = 255

            valid = (gt_cls != 255)
            acc   = np.mean(pred_cls[valid] == gt_cls[valid]) if valid.any() else 0.0

            axes[0, col].imshow(unnorm(img_t))
            axes[0, col].set_title(fname[:20], fontsize=7)
            axes[0, col].axis('off')
            axes[1, col].imshow(colorise(gt_cls))
            axes[1, col].axis('off')
            axes[2, col].imshow(colorise(pred_cls))
            axes[2, col].set_title(f'acc={acc:.3f}', fontsize=9)
            axes[2, col].axis('off')

    legend = [mpatches.Patch(color=np.array(PSCL_COLORS[k]) / 255, label=n)
              for k, n in zip([0, 1, 2, 3, 255], PSCL_NAMES)]
    fig.legend(handles=legend, loc='lower center', ncol=5, fontsize=9, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 1])
    out_path = os.path.join('/kaggle/working', 'predictions.png')
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Saved to {out_path}')